In [4]:
import random
random.seed(109)

In [5]:
class Mancala:
    # ------------------------ Helpers -------------------------- 

    # get the other player
    def _other(self, player):
        return 2 if player == 1 else 1

    # get the pit indices of the player vs the enemy 
    def _pit_indices(self, player):
        start, end = self.p1_pits_index if player == 1 else self.p2_pits_index
        return range(start, end + 1)

    # get the mancala index of the player vs the enemy 
    def _mancala_index(self, player):
        return self.p1_mancala_index if player == 1 else self.p2_mancala_index

    # Get the board index of the pit directly across from it 
    def _opposite_index(self, index):
        if index in self._pit_indices(1):
            offset = index - self.p1_pits_index[0]
            return self.p2_pits_index[1] - offset
        offset = index - self.p2_pits_index[0]
        return self.p1_pits_index[1] - offset

    # Get the winner 
    def get_winner(self):
        if not self.winning_eval():
            return None
        p1 = self.board[self.p1_mancala_index]
        p2 = self.board[self.p2_mancala_index]
        if p1 == p2:
            return 0
            
        return 1 if p1 > p2 else 2
    # ------------------------ Code ----------------------------------------- 
    def __init__(self, pits_per_player=6, stones_per_pit = 4):
        """
        The constructor for the Mancala class defines several instance variables:

        pits_per_player: This variable stores the number of pits each player has.
        stones_per_pit: It represents the number of stones each pit contains at the start of any game.
        board: This data structure is responsible for managing the Mancala board.
        current_player: This variable takes the value 1 or 2, as it's a two-player game, indicating which player's turn it is.
        moves: This is a list used to store the moves made by each player. It's structured in the format (current_player, chosen_pit).
        p1_pits_index: A list containing two elements representing the start and end indices of player 1's pits in the board data structure.
        p2_pits_index: Similar to p1_pits_index, it contains the start and end indices for player 2's pits on the board.
        p1_mancala_index and p2_mancala_index: These variables hold the indices of the Mancala pits on the board for players 1 and 2, respectively.
        """
        self.pits_per_player = pits_per_player
        self.board = [stones_per_pit] * ((pits_per_player+1) * 2)  # Initialize each pit with stones_per_pit number of stones 
        self.players = 2
        self.current_player = 1
        self.moves = []
        # Continuations 
        self.continuations = 0
        self.p1_pits_index = [0, self.pits_per_player-1]
        self.p1_mancala_index = self.pits_per_player
        self.p2_pits_index = [self.pits_per_player+1, len(self.board)-1-1]
        self.p2_mancala_index = len(self.board)-1
        
        # Zeroing the Mancala for both players
        self.board[self.p1_mancala_index] = 0
        self.board[self.p2_mancala_index] = 0

    def display_board(self):
        """
        Displays the board in a user-friendly format
        """
        player_1_pits = self.board[self.p1_pits_index[0]: self.p1_pits_index[1]+1]
        player_1_mancala = self.board[self.p1_mancala_index]
        player_2_pits = self.board[self.p2_pits_index[0]: self.p2_pits_index[1]+1]
        player_2_mancala = self.board[self.p2_mancala_index]

        print('P1               P2')
        print('     ____{}____     '.format(player_2_mancala))
        for i in range(self.pits_per_player):
            if i == self.pits_per_player - 1:
                print('{} -> |_{}_|_{}_| <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            else:    
                print('{} -> | {} | {} | <- {}'.format(i+1, player_1_pits[i], 
                        player_2_pits[-(i+1)], self.pits_per_player - i))
            
        print('         {}         '.format(player_1_mancala))
        turn = 'P1' if self.current_player == 1 else 'P2'
        print('Turn: ' + turn)
        print()  # blank line between boards
        
    def valid_move(self, pit):
        """
        Function to check if the pit chosen by the current_player is a valid move.
        """

         # Must be a whole number
        if not isinstance(pit, int):
            return False

        # Check for real pit number (1 to pits_per_player)
        if pit < 1 or pit > self.pits_per_player:
            return False

        # Must have stones in it
        index = self._pit_indices(self.current_player)[pit - 1]
        return self.board[index] > 0
        
    def random_move_generator(self):
        """
        Function to generate random valid moves with non-empty pits for the random player
        """
        options = []
        # pit numbers 1..6
        for pit in range(1, self.pits_per_player + 1):  
            # check if non empty
            if self.valid_move(pit):                       
                options.append(pit)
    
        # no legal moves 
        if len(options) == 0:       
            return None
        return random.choice(options)
    
    def play(self, pit):
        """
        This function simulates a single move made by the current player using their selected pit (1-indexed).
        Do these checks in this order:
        1. If the game is already over (winning_eval() returns True), print "GAME OVER" and take no further action.
        2. Otherwise, if the chosen pit is not a valid move for the current player, print "INVALID MOVE" and take no action.
           The current player stays the same.
        3. Otherwise, distribute the stones according to the Mancala rules, including captures, and add
           (current_player, pit) to self.moves.

        Finally, apply the continuation rule: if the last stone landed in the current player's own mancala,
        the same player moves again; otherwise switch the current player so the other player takes their turn.
        If the move ends the game, there is no extra turn.

        Returns the board (self.board).
        """

        # check if game if over 
        if self.winning_eval():
            print("GAME OVER")
            return self.board
        # Check for valid move 
        if not self.valid_move(pit):
            print("INVALID MOVE")
            return self.board
            
        # Else distribute acoording to manacla moves 

        # get whose playing 
        player = self.current_player
        player_mancala = self._mancala_index(player)
        enemy_mancala = self._mancala_index(self._other(player))
        self.moves.append((player, pit))

        # 1) get all the stones from the chosen pit 
        index = self._pit_indices(player)[pit - 1]
        stones = self.board[index]
        self.board[index] = 0

        # 2) drop 1 stone per pit cc skipping opponents mancala 
        while stones > 0:
            index = (index + 1) % len(self.board)
            if index == enemy_mancala:
                continue
            self.board[index] += 1
            stones -= 1

        # 3) Capture (if u land in empty pit cature the stone and enemy pit stones) 
        if index in self._pit_indices(player) and self.board[index] == 1:
            opposite = self._opposite_index(index)
            self.board[player_mancala] += self.board[index] + self.board[opposite]
            self.board[index] = 0
            self.board[opposite] = 0

        # 4) check if this won the game 
        game_over = self.winning_eval()
        if game_over:
            winner = self.get_winner()
            if winner == 0:
                print("It's a tie!")
            else:
                print(f"Player {winner} wins!")

        # 5) continuation rule 
        if index == player_mancala and not game_over:
            self.continuations += 1 
            return self.board 
            
        self.current_player = self._other(player)

        # write your code here
        return self.board
    
    def winning_eval(self):
        """
        Function to verify if the game board has reached the winning state.
        Hint: If either of the players' pits are all empty, then it is considered a winning state.
        When the game is over, move any stones left on a player's side into that player's mancala.

        Returns True if the game is over and False otherwise (always a bool, never None or the winner).
        To report the winner, compare the two mancalas after winning_eval() returns True.
        """
        p1_empty = all(self.board[i] == 0 for i in self._pit_indices(1))
        p2_empty = all(self.board[i] == 0 for i in self._pit_indices(2))
        if not (p1_empty or p2_empty):
            return False

        for player in (1, 2):
            mancala = self._mancala_index(player)
            for i in self._pit_indices(player):
                self.board[mancala] += self.board[i]
                self.board[i] = 0
        return True     


In [9]:
# ------------------------------- Driver Code ------------------------------
# Create a 6-pit (12 pits + 2 mancalas), 4-stone game,
# Both players will be playing random moves 


# ---------------------------- Helper Functions ----------------------------
def prct(n):
    # for calculating percent when nu Gmaes isnt always 100
    return 100 * n / NUM_GAMES

# Constants 
NUM_GAMES = 100
wins = {1: 0, 2: 0}
ties = 0 
turns = 0
continuations = 0

# -------- Display the board setup used for the 100 games ------------------
game = Mancala(pits_per_player=6, stones_per_pit=4)
game.display_board


# --------------------------------------------------------------------------

# Driver loop
for _ in range(NUM_GAMES):
    # 1) Set the game board 
    game = Mancala(pits_per_player=6, stones_per_pit=4)

    # 2) Play the game until theres a winner 
    while not game.winning_eval():
        game.play(game.random_move_generator())
        
    # 3) Determine the winner and update scores
    winner = game.get_winner()
    if winner == 0:
        ties += 1
    else:
        wins[winner] += 1
        
    # 4) update totals 
    turns += len(game.moves)
    continuations += game.continuations

print(f"Random vs. Random, {NUM_GAMES} games\n")
for p in (1, 2):
    other = 2 if p == 1 else 1
    print(f"Player {p}: won {wins[p]} ({prct(wins[p]):.1f}%), "
          f"lost {wins[other]} ({prct(wins[other]):.1f}%), "
          f"tied {ties} ({prct(ties):.1f}%)")


print(f"\nAverage turns per game: {turns / NUM_GAMES:.2f}")
print(f"Average continuations per game: {continuations / NUM_GAMES:.2f}")


Player 2 wins!
Player 1 wins!
Player 2 wins!
It's a tie!
Player 2 wins!
It's a tie!
Player 2 wins!
Player 2 wins!
Player 2 wins!
It's a tie!
Player 1 wins!
Player 2 wins!
Player 2 wins!
Player 2 wins!
Player 1 wins!
It's a tie!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
It's a tie!
Player 1 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 1 wins!
It's a tie!
Player 2 wins!
Player 2 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 1 wins!
Player 2 wins!
Player 2 wins!
Player 1 wins!
Player 1 wins!
Player 2 wins!
Player 1 wins!
Player 2 wins!
It's a tie!
Player 1 wins!
Player 1 wins!
Player 1 wins!
It's a tie!
Player 1 wins!
Player 1 wins!
Play